In [1]:
%pip install pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


## Sección A: carga y Parquet crudo

Celda 1: rutas, archivos y verificación

In [2]:
import shutil
from pathlib import Path
import pandas as pd

BASE    = Path(r"C:\Users\fsociety\Videos\Proyecto_EEVV")   # <- pega aquí tu ruta
ORIGEN  = BASE / "datos" / "extraido"   # aquí van los 6 CSV
DESTINO = BASE / "datos" / "crudo"      # aquí se escribe el Parquet

GRUPOS = {
    "defunciones_no_fetales": [
        "defuncion_no_fetal_2022.csv",
        "defuncion_no_fetal_2023.csv",
        "defunciones_no_fetales_2024.csv",
    ],
    "nacimientos": [
        "nacimientos_2022.csv",
        "nacimientos_2023.csv",
        "nacimientos_2024.csv",
    ],
}

faltan = [n for fs in GRUPOS.values() for n in fs if not (ORIGEN / n).exists()]
print("Archivos que faltan:", faltan if faltan else "ninguno")

Archivos que faltan: ninguno


Celda 2: unión de columnas

In [3]:
def columnas_union(archivos):
    union = []
    for nombre in archivos:
        cols = pd.read_csv(ORIGEN / nombre, dtype=str, encoding="cp1252", nrows=0).columns
        for c in cols.str.strip().str.upper():
            if c not in union:
                union.append(c)
    return union

Celda 3: ingesta a Parquet

In [4]:
for grupo, archivos in GRUPOS.items():
    union = columnas_union(archivos)
    salida = DESTINO / grupo
    shutil.rmtree(salida, ignore_errors=True)   # evita duplicar datos si repites la celda

    for nombre in archivos:
        df = pd.read_csv(ORIGEN / nombre, dtype=str, encoding="cp1252",
                         keep_default_na=False, na_values=[""])
        df.columns = df.columns.str.strip().str.upper()

        df["es_duplicado_exacto"] = df.duplicated(keep="first")
        df["archivo_origen"] = nombre

        df = df.reindex(columns=union + ["es_duplicado_exacto", "archivo_origen"])
        texto = [c for c in df.columns if c != "es_duplicado_exacto"]
        df[texto] = df[texto].astype("string")

        df.to_parquet(salida, engine="pyarrow", partition_cols=["ANO"], index=False)
        print(f"{nombre}: {len(df):,} filas, {int(df['es_duplicado_exacto'].sum()):,} duplicados")
        del df

defuncion_no_fetal_2022.csv: 287,251 filas, 101 duplicados
defuncion_no_fetal_2023.csv: 268,411 filas, 224 duplicados
defunciones_no_fetales_2024.csv: 275,778 filas, 102 duplicados
nacimientos_2022.csv: 573,625 filas, 584 duplicados
nacimientos_2023.csv: 515,549 filas, 1,568 duplicados
nacimientos_2024.csv: 453,901 filas, 2,906 duplicados


Celda 4: validación

In [5]:
esperado = {
    "defunciones_no_fetales": {"2022": 287251, "2023": 268411, "2024": 275778},
    "nacimientos":            {"2022": 573625, "2023": 515549, "2024": 453901},
}
for grupo, esp in esperado.items():
    df = pd.read_parquet(DESTINO / grupo)
    real = df["ANO"].astype(str).value_counts().to_dict()
    print(grupo, "OK" if real == esp else f"REVISAR {real}", "| columnas:", df.shape[1])

defunciones_no_fetales OK | columnas: 64
nacimientos OK | columnas: 42


## Sección B: Diccionario y renombrado

Celda 5: leer el diccionario

In [6]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [7]:
DICCIONARIO = BASE / "documentacion" / "diccionario_variables_eevv.xlsx"

dic = pd.read_excel(DICCIONARIO, sheet_name="variables", dtype=str)
print("Diccionario:", dic.shape)
print(dic.groupby("grupo").size())

Diccionario: (106, 13)
grupo
defunciones_no_fetales    64
nacimientos               42
dtype: int64


Celda 6: validar el diccionario

In [8]:
import re
import pyarrow.dataset as ds

patron = re.compile(r"^[a-z][a-z0-9_]*$")

for grupo in GRUPOS:
    d = dic[dic["grupo"] == grupo]
    cols = set(ds.dataset(DESTINO / grupo, partitioning="hive").schema.names)

    faltan    = cols - set(d["nombre_original"])
    sobran    = set(d["nombre_original"]) - cols
    repetidos = d.loc[d["nombre_nuevo"].duplicated(), "nombre_nuevo"].tolist()
    invalidos = [n for n in d["nombre_nuevo"] if not patron.match(str(n))]

    print(f"{grupo}: faltan={sorted(faltan)} sobran={sorted(sobran)} "
          f"repetidos={repetidos} invalidos={invalidos}")
    assert not (faltan or sobran or repetidos or invalidos), f"Revisar el diccionario: {grupo}"

defunciones_no_fetales: faltan=[] sobran=[] repetidos=[] invalidos=[]
nacimientos: faltan=[] sobran=[] repetidos=[] invalidos=[]


Celda 7: renombrar

In [9]:
def renombrar(df, grupo):
    d = dic[dic["grupo"] == grupo]
    mapa = dict(zip(d["nombre_original"], d["nombre_nuevo"]))
    return df.rename(columns=mapa)

datos = {}
for grupo in GRUPOS:
    datos[grupo] = renombrar(pd.read_parquet(DESTINO / grupo), grupo)
    print(grupo, datos[grupo].shape)

datos["defunciones_no_fetales"][["anio", "mes", "sexo", "grupo_edad_detallado",
                                 "dpto_residencia_cod", "causa_lista_667"]].head()

defunciones_no_fetales (831440, 64)
nacimientos (1543075, 42)


,anio,mes,sexo,grupo_edad_detallado,dpto_residencia_cod,causa_lista_667
0,2022,12,2,21,15,307
1,2022,12,1,27,20,306
2,2022,06,2,26,05,106
3,2022,08,1,25,52,214
4,2022,04,1,20,05,510


## Sección C: Tipos de dato y sin información

Celda 8: asignar tipos según el diccionario

In [10]:
def codigos_sin_info(valor):
    if pd.isna(valor) or "verificar" in valor:
        return []
    return [c.strip() for c in valor.split(",") if c.strip()]

def tipar(df, grupo):
    reglas = dic[dic["grupo"] == grupo]
    reporte = []
    for _, r in reglas.iterrows():
        col, tipo = r["nombre_nuevo"], r["tipo_sugerido"]
        s = df[col]
        nulos_antes = int(s.isna().sum())
        a_nulo = 0

        if tipo == "entero":
            s = s.astype("string")
            sin_info = codigos_sin_info(r["codigos_sin_info"])
            if sin_info:
                marca = s.isin(sin_info)
                a_nulo = int(marca.sum())
                s = s.mask(marca)
            s = pd.to_numeric(s, errors="coerce").astype("Int64")
        elif tipo == "fecha":
            s = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
        elif tipo == "categoría":
            s = s.astype("category")
        elif tipo == "indicador" and s.dtype != bool:
            s = s.eq("1").fillna(False).astype(bool)
        # "código" y "texto libre" se quedan como texto

        df[col] = s
        perdidos = int(s.isna().sum()) - nulos_antes - a_nulo if tipo in ("entero", "fecha") else 0
        reporte.append({"columna": col, "tipo": tipo, "sin_info_a_nulo": a_nulo,
                        "perdidos_en_conversion": perdidos})
    return df, pd.DataFrame(reporte)

reportes = {}
for grupo in GRUPOS:
    datos[grupo], reportes[grupo] = tipar(datos[grupo], grupo)
    rep = reportes[grupo]
    print(f"\n== {grupo}")
    print(datos[grupo].dtypes.astype(str).value_counts().to_string())
    print(rep[(rep.sin_info_a_nulo > 0) | (rep.perdidos_en_conversion != 0)].to_string(index=False))


== defunciones_no_fetales
category    30
string      18
Int64        8
bool         8
              columna   tipo  sin_info_a_nulo  perdidos_en_conversion
  hijos_nacidos_vivos entero             1217                       0
hijos_nacidos_muertos entero              135                       0
   ultimo_curso_madre entero             5335                       0

== nacimientos
category          22
Int64             10
string             8
datetime64[us]     1
bool               1
                 columna   tipo  sin_info_a_nulo  perdidos_en_conversion
num_consultas_prenatales entero            15600                       0
             apgar_1_min entero            38507                       0
             apgar_5_min entero            38507                       0
      ultimo_curso_madre entero            55202                       0
     hijos_nacidos_vivos entero            15317                       0
           num_embarazos entero            15351                       0
 

Celda 9: revisar los rangos de los enteros

In [11]:
for grupo in GRUPOS:
    enteros = [c for c in datos[grupo].columns if str(datos[grupo][c].dtype) == "Int64"]
    print(f"\n== {grupo}: mínimo y máximo de las columnas enteras")
    print(datos[grupo][enteros].agg(["min", "max"]).T.to_string())


== defunciones_no_fetales: mínimo y máximo de las columnas enteras
                         min   max
mes                        1    12
hora                       0    23
minutos                    0    59
ultimo_curso_fallecido     0    99
hijos_nacidos_vivos        1    12
hijos_nacidos_muertos      0     9
ultimo_curso_madre         0    13
anio                    2022  2024

== nacimientos: mínimo y máximo de las columnas enteras
                           min   max
mes                          1    12
num_consultas_prenatales     0    25
apgar_1_min                  1    10
apgar_5_min                  1    10
ultimo_curso_madre           0    13
hijos_nacidos_vivos          1    17
num_embarazos                1    20
edad_padre                  12    85
ultimo_curso_padre           0    13
anio                      2022  2024


## Sección D: Filtro de primera infancia y región Pacífica

Celda 10: constantes y filtro de defunciones

In [12]:
DPTOS_PACIFICO = {"27": "Chocó", "76": "Valle del Cauca", "19": "Cauca", "52": "Nariño"}
EDAD_MENOR_5 = [f"{i:02d}" for i in range(0, 9)]          # códigos 00 a 08
GRUPO_EDAD = {**dict.fromkeys(["00", "01", "02", "03"], "Neonatal"),
              **dict.fromkeys(["04", "05", "06"], "Posneonatal"),
              **dict.fromkeys(["07", "08"], "1 a 4 años")}

d = datos["defunciones_no_fetales"]
menor_5     = d["grupo_edad_detallado"].isin(EDAD_MENOR_5)
menor_5_ed2 = d["grupo_edad_agrupado"].isin(["01", "02"])
en_region   = d["dpto_residencia_cod"].isin(list(DPTOS_PACIFICO))

assert (menor_5 == menor_5_ed2).all(), "GRU_ED1 y GRU_ED2 no coinciden: revisar códigos de edad"

print("Defunciones totales:                ", f"{len(d):,}")
print("Menores de 5 (todo el país):        ", f"{menor_5.sum():,}")
print("  sin departamento de residencia:   ", f"{(menor_5 & d['dpto_residencia_cod'].isna()).sum():,}")
print("Menores de 5 residentes en Pacífico:", f"{(menor_5 & en_region).sum():,}")

defun = d[menor_5 & en_region].copy()
defun["grupo_edad_analisis"]     = defun["grupo_edad_detallado"].map(GRUPO_EDAD).astype("category")
defun["dpto_residencia_nombre"]  = defun["dpto_residencia_cod"].map(DPTOS_PACIFICO).astype("category")
defun["mun_residencia_divipola"] = defun["dpto_residencia_cod"] + defun["mun_residencia_cod"]

Defunciones totales:                 831,440
Menores de 5 (todo el país):         21,417
  sin departamento de residencia:    650
Menores de 5 residentes en Pacífico: 3,217


Celda 11: filtro de nacimientos

In [13]:
n = datos["nacimientos"]
nac = n[n["dpto_residencia_cod"].isin(list(DPTOS_PACIFICO))].copy()
nac["dpto_residencia_nombre"]  = nac["dpto_residencia_cod"].map(DPTOS_PACIFICO).astype("category")
nac["mun_residencia_divipola"] = nac["dpto_residencia_cod"] + nac["mun_residencia_cod"]

print("Nacimientos totales:", f"{len(n):,}")
print("  sin departamento de residencia:", f"{n['dpto_residencia_cod'].isna().sum():,}")
print("Nacimientos de madres residentes en Pacífico:", f"{len(nac):,}")

Nacimientos totales: 1,543,075
  sin departamento de residencia: 24,401
Nacimientos de madres residentes en Pacífico: 211,367


Celda 12: resumen del subconjunto

In [14]:
print(pd.crosstab(defun["grupo_edad_analisis"], defun["anio"], margins=True, margins_name="Total"), "\n")
print(pd.crosstab(defun["dpto_residencia_nombre"], defun["anio"], margins=True, margins_name="Total"), "\n")
print(pd.crosstab(nac["dpto_residencia_nombre"], nac["anio"], margins=True, margins_name="Total"))

anio                 2022  2023  2024  Total
grupo_edad_analisis                         
1 a 4 años            242   233   197    672
Neonatal              571   487   466   1524
Posneonatal           356   368   297   1021
Total                1169  1088   960   3217 

anio                    2022  2023  2024  Total
dpto_residencia_nombre                         
Cauca                    223   217   144    584
Chocó                    273   269   247    789
Nariño                   193   169   159    521
Valle del Cauca          480   433   410   1323
Total                   1169  1088   960   3217 

anio                     2022   2023   2024   Total
dpto_residencia_nombre                             
Cauca                   16375  14115  12600   43090
Chocó                    7265   6642   5864   19771
Nariño                  14078  12610  11387   38075
Valle del Cauca         40862  36514  33055  110431
Total                   78580  69881  62906  211367


## Sección E: guardar la capa procesada

Celda 13: guardar

In [15]:
PROCESADO = BASE / "datos" / "procesado"
PROCESADO.mkdir(parents=True, exist_ok=True)

SALIDAS = {
    "defunciones_primera_infancia_pacifico.parquet": defun,
    "nacimientos_pacifico.parquet": nac,
}

for nombre, tabla in SALIDAS.items():
    tabla.to_parquet(PROCESADO / nombre, engine="pyarrow", index=False)
    tamano_kb = (PROCESADO / nombre).stat().st_size / 1024
    print(f"{nombre}: {len(tabla):,} filas, {tabla.shape[1]} columnas, {tamano_kb:,.0f} KB")

defunciones_primera_infancia_pacifico.parquet: 3,217 filas, 67 columnas, 155 KB
nacimientos_pacifico.parquet: 211,367 filas, 44 columnas, 3,163 KB


Celda 14: verificar

In [16]:
for nombre, original in SALIDAS.items():
    original = original.reset_index(drop=True)
    leido = pd.read_parquet(PROCESADO / nombre)

    tipos_distintos = original.dtypes.astype(str) != leido.dtypes.astype(str)
    print(f"{nombre}: {len(leido):,} filas | {leido.shape[1]} columnas | "
          f"tipos distintos: {list(original.columns[tipos_distintos])}")

    pd.testing.assert_frame_equal(original, leido, check_dtype=False, check_categorical=False)
    print("   valores idénticos: OK")

defunciones_primera_infancia_pacifico.parquet: 3,217 filas | 67 columnas | tipos distintos: []
   valores idénticos: OK
nacimientos_pacifico.parquet: 211,367 filas | 44 columnas | tipos distintos: []
   valores idénticos: OK
